# Comparing initial and final distributions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
from pypopsyn.simulator.configuration import cfg
import pypopsyn.simulator.basics.constants as const
import utilities.plot_settings

Import the initial population file.

In [ ]:
data_i = pd.read_pickle(
    "../examples/data/simulation_full_example/initial_population.pkl.gz",
    compression="gzip",
)
data_i.head()

In [ ]:
age = data_i["age"]["[yr]"].to_numpy()
x_i = data_i["x"]["[kpc]"].to_numpy()
y_i = data_i["y"]["[kpc]"].to_numpy()
z_i = data_i["z"]["[kpc]"].to_numpy()
vk_r = data_i["vk_r"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
vk_phi = data_i["vk_phi"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
vk_z = data_i["vk_z"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
v_orb = data_i["v_orb"]["[kpc yr^-1]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
B_i = data_i["B"]["[G]"].to_numpy()
chi_i = data_i["chi"]["[rad]"].to_numpy()
P_i = data_i["P"]["[s]"].to_numpy()
P_dot_i = data_i["P_dot"]["[s yr^-1]"].to_numpy()

Import the final population file.

In [ ]:
data_f = pd.read_pickle(
    "../examples/data/simulation_full_example/final_population.pkl.gz",
    compression="gzip",
)
data_f.head()

In [ ]:
x_f = data_f["x"]["[kpc]"].to_numpy()
y_f = data_f["y"]["[kpc]"].to_numpy()
z_f = data_f["z"]["[kpc]"].to_numpy()
RA_f = data_f["RA"]["[deg]"].to_numpy()
DEC_f = data_f["DEC"]["[deg]"].to_numpy()
pm_RA_f = data_f["pm_RA"]["[mas yr^-1]"].to_numpy()
pm_DEC_f = data_f["pm_DEC"]["[mas yr^-1]"].to_numpy()
v_r_f = data_f["v_r"]["[km s^-1]"].to_numpy()
v_phi_f = data_f["v_phi"]["[km s^-1]"].to_numpy()
v_z_f = data_f["v_z"]["[km s^-1]"].to_numpy()
dist_f = data_f["d"]["[kpc]"].to_numpy()
B_f = data_f["B"]["[G]"].to_numpy()
chi_f = data_f["chi"]["[rad]"].to_numpy()
P_f = data_f["P"]["[s]"].to_numpy()
P_dot_f = data_f["P_dot"]["[s yr^-1]"].to_numpy()
L_radio = data_f["L_radio"]["[erg s^-1 Hz^-1]"].to_numpy()
S_radio_Jy = data_f["S_radio"]["[Jy]"].to_numpy()
w_int = data_f["w_int"]["[s]"].to_numpy()
intercepted_radio = data_f["intercepted_radio"][" "].to_numpy(dtype=bool)
detected_radio_PMPS = data_f["detected_radio_PMPS"][" "].to_numpy(dtype=bool)
detected_radio_SMPS = data_f["detected_radio_SMPS"][" "].to_numpy(dtype=bool)

detected_radio = detected_radio_PMPS | detected_radio_SMPS

Plot of the spatial distribution in galactocentric coordinates.
In the background, in cyan the initial distribution of all the stars, in orange the final distribution of all evolved stars, in blue the initial position of stars that have been detected, and in red the final position of stars that are detected.

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x_i,
    y_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    x_f,
    y_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    x_i[detected_radio],
    y_i[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1,
    rasterized=True
)
ax.plot(
    x_f[detected_radio],
    y_f[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.,
    rasterized=True
)

ax.plot(0.0, 8.3, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x_i,
    z_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True,
)
ax.plot(
    x_f,
    z_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True,
)
ax.plot(
    x_i[detected_radio],
    z_i[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.,
    rasterized=True,
)
ax.plot(
    x_f[detected_radio],
    z_f[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.,
    rasterized=True
)

ax.plot(0.0, 0.02, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-5.0, 5.0)

plt.show()

Galactocentric radius distribution.

In [ ]:
r_i = np.sqrt(x_i**2 + y_i**2)
r_f = np.sqrt(x_f**2 + y_f**2)
r_bins = np.linspace(0.0, 30.0, 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    r_i,
    bins=r_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    alpha=1,
    label="Initial all"
)
ax.hist(
    r_i[detected_radio],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Initial detected"
)
ax.hist(
    r_f,
    bins=r_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    alpha=1,
    label="Final all"
)
ax.hist(
    r_f[detected_radio],
    bins=r_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="Final detected",
)

plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"Number of NSs")
plt.xlim(0.0, 30.0)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Galactic height distribution.

In [ ]:
z_bins = np.linspace(0.0, 5.0, 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    z_i,
    bins=z_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    alpha=1,
    label="Initial all"
)
ax.hist(
    z_i[detected_radio],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Initial detected"
)
ax.hist(
    z_f,
    bins=z_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    alpha=1,
    label="Final all"
)
ax.hist(
    z_f[detected_radio],
    bins=z_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label="Final detected",
)

plt.xlabel(r"$z$ [kpc]")
plt.ylabel(r"Number of NSs")
plt.xlim(0.0, 5.0)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Total velocity magnitude distribution.

In [ ]:
v_tot_i = np.sqrt(vk_r**2 + (vk_phi + v_orb)**2 + vk_z**2)
v_tot_f = np.sqrt(v_r_f**2 + v_phi_f**2 + v_z_f**2)

fig, ax = plt.subplots(figsize=(15,8))
v_bins = np.linspace(0,1600.,31)  

ax.hist(
    v_tot_i,
    bins=v_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    alpha=1,
    label=r"Initial all",
)
ax.hist(
    v_tot_i[detected_radio],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Initial detected",
)
ax.hist(
    v_tot_f,
    bins=v_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    alpha=1,
    label=r"final all",
)
ax.hist(
    v_tot_f[detected_radio],
    bins=v_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"Final detected",
)

ax.set_xlabel(r"Total velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.yscale('log')
plt.ylim(0.1, 1.e5)
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Spin period distribution.

In [ ]:
P_bins = np.logspace(-6., 3., 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    P_i,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    label="Initial all",
)
ax.hist(
    P_i[detected_radio],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Initial detected",
)
ax.hist(
    P_f,
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    label="Final all",
)
ax.hist(
    P_f[detected_radio],
    bins=P_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Final detected",
)
plt.xlabel(r"$P$ [s]")
plt.ylabel(r"Number of NSs")
#plt.xlim(0., 50.0)
plt.ylim(0.1, 2.e5)
plt.xscale('log')
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Magnetic field distribution.

In [ ]:
B_log10_bins = np.linspace(7.0, 17.0, 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    np.log10(B_i),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    label="Initial all",
)
ax.hist(
    np.log10(B_i[detected_radio]),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Initial detected",
)
ax.hist(
    np.log10(B_f),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    label="Final all",
)
ax.hist(
    np.log10(B_f[detected_radio]),
    bins=B_log10_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Final detected",
)

plt.xlabel(r"log$_{10} B$ [G]")
plt.ylabel(r"Number of NSs")
plt.ylim(0.1, 1.e5)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

Inclination angle distribution.

In [ ]:
chi_bins = np.linspace(0, np.pi / 2, 31)

fig, ax = plt.subplots(figsize=(15,8))

ax.hist(
    chi_i,
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    ls=":",
    label="Initial all",
)
ax.hist(
    chi_i[detected_radio],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    label="Initial detected",
)
ax.hist(
    chi_f,
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    ls=":",
    label="Final all",
)
ax.hist(
    chi_f[detected_radio],
    bins=chi_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    label="Final detected",
)

plt.xlabel(r"$\chi$ [rad]")
plt.ylabel(r"Number of NSs")
plt.xlim(0., np.pi / 2)
plt.ylim(0.1, 1.e5)
plt.yscale('log')
plt.legend(bbox_to_anchor=(1.05, 1), frameon=False, loc=0, fontsize=20)

plt.show()

## Relationships between independent parameters

Magnetic field vs spin period.

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    P_i,
    np.log10(B_i),
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    P_f,
    np.log10(B_f),
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    P_i[detected_radio],
    np.log10(B_i[detected_radio]),
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.,
    rasterized=True
)
ax.plot(
    P_f[detected_radio],
    np.log10(B_f[detected_radio]),
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.,
    rasterized=True
)

ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"log$_{10} B$ [G]")
plt.xscale('log') 

plt.show()

As a reference we show also the $P-\dot{P}$ diagram.

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    P_i,
    P_dot_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    P_f,
    P_dot_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    P_i[detected_radio],
    P_dot_i[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.,
    rasterized=True
)
ax.plot(
    P_f[detected_radio],
    P_dot_f[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.,
    rasterized=True
)

ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"$\dot{P}$")
plt.xscale('log') 
plt.yscale('log') 

plt.show()

Inclination angle vs period.

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    P_i,
    chi_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    P_f,
    chi_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    P_i[detected_radio],
    chi_i[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.,
    rasterized=True
)
ax.plot(
    P_f[detected_radio],
    chi_f[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.,
    rasterized=True
)

ax.set_xlabel(r"$P$ [s]")
ax.set_ylabel(r"$\chi$ [rad]")
plt.xscale('log') 

plt.show()

Inclination angle vs Bfield.

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    B_i,
    chi_i,
    linestyle="None",
    marker="o",
    color="tab:cyan",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    B_f,
    chi_f,
    linestyle="None",
    marker="o",
    color="tab:orange",
    markersize=1,
    alpha=0.1,
    rasterized=True
)
ax.plot(
    B_i[detected_radio],
    chi_i[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=2,
    alpha=1.,
    rasterized=True
)
ax.plot(
    B_f[detected_radio],
    chi_f[detected_radio],
    linestyle="None",
    marker="o",
    color="tab:red",
    markersize=2,
    alpha=1.,
    rasterized=True
)

ax.set_xlabel(r"$B$ [G]")
ax.set_ylabel(r"$\chi$ [rad]")
plt.xscale('log') 

plt.show()